# EDA — Part A: Basic Inspection

**Goal:** Know the data before modelling it.  
This notebook characterises every FER-2013 CSV: schema differences between files,
shape, dtypes, missing values, pixel value ranges, label distribution, and Usage split counts.

Every anomaly found here feeds directly into the cleaning and preprocessing stages
(`config.yaml → cleaning.*` and `preprocessing.*`).

**File inventory** (all live in `data/` after the download stage):

| File | Columns | Role in pipeline |
|---|---|---|
| `train.csv` | `emotion`, `pixels` | Flat training rows — no split column |
| `test.csv` | `pixels` | Unlabelled; not used by this pipeline |
| `test_with_emotions.csv` | `emotion`, `pixels` | Held-out evaluation set |
| `icml_face_data.csv` | `emotion`, `usage`, `pixels` | **Primary CSV** — has the split column |
| `fer2013/fer2013.csv` | `emotion`, `pixels`, `usage` | Same schema as `icml_face_data.csv` |

`Fer2013Fetcher` targets `icml_face_data.csv` (`config.yaml → data.primary_csv`)  
because it is the only top-level file with a `usage` column to separate
Training / PublicTest / PrivateTest.

## 0. Setup

In [ ]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path().resolve().parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.emotion_detector.utils.config import load_config
from src.emotion_detector.utils.logging import setup_logging
from src.emotion_detector.data.fer2013 import Fer2013Fetcher

cfg = load_config(ROOT / "config.yaml")
setup_logging(cfg)

DATA_DIR = ROOT / cfg["paths"]["data_dir"]

FILES = {
    "primary"  : DATA_DIR / cfg["data"]["primary_csv"],   # icml_face_data.csv
    "train"    : DATA_DIR / cfg["data"]["train_csv"],      # train.csv
    "test_eval": DATA_DIR / cfg["data"]["test_csv"],       # test_with_emotions.csv
}

for label, path in FILES.items():
    status = "✓" if path.exists() else "✗ MISSING"
    print(f"{status}  {label:12} → {path.name}")

## 1. Schema survey — what columns does each file actually have?

Each file has a different schema. This cell loads just the header row
(0 data rows) so we can compare without reading 35 k rows.

In [ ]:
for label, path in FILES.items():
    header = pd.read_csv(path, nrows=0)
    print(f"{path.name}")
    print(f"  columns : {list(header.columns)}")
    print()

## 2. Load the primary CSV (`icml_face_data.csv`)

This is the file that drives `Fer2013Fetcher` — it has all rows plus the `usage`
column that separates Training / PublicTest / PrivateTest.

In [ ]:
df = pd.read_csv(FILES["primary"])

# Strip leading/trailing whitespace from column names — the real CSV
# ships headers like ' Usage' and ' pixels' with a leading space.
df.columns = df.columns.str.strip()

# Normalise the split column: real data ships as lowercase 'usage'
usage_col = next((c for c in df.columns if c.lower() == "usage"), None)
if usage_col and usage_col != "Usage":
    df = df.rename(columns={usage_col: "Usage"})

print(f"Shape   : {df.shape}")
print(f"Columns : {list(df.columns)}")
df.head()

## 3. Schema — dtypes

In [ ]:
df.info()

In [ ]:
print("Dtypes:")
print(df.dtypes)
print()
print("Expected:")
print("  emotion  → int64             (label 0–6)")
print("  pixels   → object or str     (space-separated pixel string; str = StringDtype in pandas 3.x)")
print("  Usage    → object or str     (Training / PublicTest / PrivateTest)")
print()
print("Note: 'object' and 'str' both mean string data.")
print("      pandas 3.x uses StringDtype (displayed as 'str') instead of the")
print("      legacy object dtype. Both are handled identically by str methods and np.fromstring.")

## 4. Summary statistics

In [ ]:
df.describe(include="all")

## 5. Missing value audit

In [ ]:
nan_counts = df.isna().sum()
print("NaN counts per column:")
print(nan_counts)
print()
if nan_counts.sum() == 0:
    print("✓ No NaN values found.")
else:
    print(f"⚠ {nan_counts.sum()} NaN values — see cleaning stage.")

In [ ]:
empty_pixels = df["pixels"].str.strip().eq("").sum()
print(f"Empty / whitespace-only pixels rows: {empty_pixels}")
if empty_pixels == 0:
    print("✓ No empty pixel strings.")

## 6. Pixel-count sanity check

Every row must encode exactly **2 304 values** (48 × 48).

In [ ]:
pixel_counts = df["pixels"].str.split().str.len()

print("Pixel-count distribution:")
print(pixel_counts.value_counts().sort_index())
print()

bad_rows = df[pixel_counts != 2304]
print(f"Rows with pixel count ≠ 2304: {len(bad_rows)}")
if len(bad_rows) == 0:
    print("✓ All rows have exactly 2304 pixel values.")
else:
    print("⚠ Malformed rows:")
    print(bad_rows[["emotion", "Usage"]].assign(pixel_count=pixel_counts[bad_rows.index]))

## 7. Pixel intensity statistics

`Fer2013Fetcher` parses the pixel strings into `(N, 48, 48)` uint8 arrays.
We compute intensity stats on the Training split.

In [ ]:
fetcher = Fer2013Fetcher(cfg)

images_train, labels_train = fetcher.fetch("Training")
images_val,   labels_val   = fetcher.fetch("PublicTest")
images_test,  labels_test  = fetcher.fetch("PrivateTest")

print(f"Training   — images: {images_train.shape}, labels: {labels_train.shape}")
print(f"Validation — images: {images_val.shape},   labels: {labels_val.shape}")
print(f"Test       — images: {images_test.shape},  labels: {labels_test.shape}")

In [ ]:
px = images_train.astype(np.float32)
print("Training pixel intensity statistics:")
print(f"  dtype   : {images_train.dtype}")
print(f"  min     : {px.min():.1f}")
print(f"  max     : {px.max():.1f}")
print(f"  mean    : {px.mean():.2f}")
print(f"  std     : {px.std():.2f}")
print(f"  median  : {np.median(px):.1f}")
print()
if px.min() >= 0 and px.max() <= 255:
    print("✓ All pixel values in valid range [0, 255].")
else:
    print("⚠ Out-of-range pixel values detected.")

## 8. Label distribution

In [ ]:
EMOTION_LABELS = {
    0: "Angry", 1: "Disgust", 2: "Fear",
    3: "Happy", 4: "Sad",     5: "Surprise", 6: "Neutral",
}

label_counts = df["emotion"].value_counts().sort_index()
label_pct    = label_counts / len(df) * 100

print("Label distribution (full icml_face_data.csv):")
print(f"{'Code':<6} {'Label':<12} {'Count':>7} {'%':>7}")
print("-" * 36)
for code, count in label_counts.items():
    print(f"{code:<6} {EMOTION_LABELS[code]:<12} {count:>7,} {label_pct[code]:>6.1f}%")
print("-" * 36)
print(f"{'Total':<18} {len(df):>7,} {'100.0%':>7}")
print()

out_of_range = df[~df["emotion"].between(0, 6)]
print(f"Out-of-range labels : {len(out_of_range)}")
if len(out_of_range) == 0:
    print("✓ All labels in [0, 6].")

## 9. Usage split counts

In [ ]:
usage_counts = df["Usage"].value_counts()
usage_pct    = usage_counts / len(df) * 100

print("Usage split distribution:")
print(f"{'Split':<16} {'Count':>7} {'%':>7}")
print("-" * 33)
for split, count in usage_counts.items():
    print(f"{split:<16} {count:>7,} {usage_pct[split]:>6.1f}%")
print("-" * 33)
print(f"{'Total':<16} {len(df):>7,} {'100.0%':>7}")
print()

unexpected = set(df["Usage"].unique()) - {"Training", "PublicTest", "PrivateTest"}
if not unexpected:
    print("✓ Only expected Usage values present.")
else:
    print(f"⚠ Unexpected Usage values: {unexpected}")

## 10. Per-split label distribution

Checks whether class imbalance is consistent across splits.

In [ ]:
split_label = (
    df.groupby(["Usage", "emotion"])
    .size()
    .unstack(fill_value=0)
    .rename(columns=EMOTION_LABELS)
)
split_label

## 11. train.csv schema check

`train.csv` has only `emotion` and `pixels` — no `Usage` column.
It contains all training rows as a flat file.  
We confirm its shape and verify it has no NaN.

In [ ]:
df_train = pd.read_csv(FILES["train"])
df_train.columns = df_train.columns.str.strip()
print(f"train.csv — shape: {df_train.shape}")
print(f"Columns  : {list(df_train.columns)}")
print(f"NaN      : {df_train.isna().sum().to_dict()}")
df_train.head(3)

## 12. test_with_emotions.csv schema check

This is the final held-out evaluation set — touched only once, after training.

In [ ]:
df_eval = pd.read_csv(FILES["test_eval"])
df_eval.columns = df_eval.columns.str.strip()
print(f"test_with_emotions.csv — shape: {df_eval.shape}")
print(f"Columns  : {list(df_eval.columns)}")
print(f"NaN      : {df_eval.isna().sum().to_dict()}")
df_eval.head(3)

## 13. Findings summary

Fill in the **Result** column after running on the actual dataset.
Each anomaly that requires a fix must become a named, switchable option in `config.yaml`
(CONTRIBUTING.md §3 — Ablation-Driven Architecture).

| Check | Result | Action |
|---|---|---|
| NaN values in icml_face_data.csv | — | — |
| Empty pixel strings | — | — |
| Pixel count ≠ 2304 | — | — |
| Pixel range [0, 255] | — | — |
| Label range [0, 6] | — | — |
| Unexpected Usage values | — | — |
| Class imbalance | Disgust ≈ 1.5 % vs Happy ≈ 25 % | `cleaning.imbalance_strategy: class_weight` |
| train.csv has no Usage column | Confirmed — flat file only | Use `icml_face_data.csv` as primary_csv |